In [ ]:
!pip install -q transformers datasets evaluate seqeval accelerate

In [ ]:
import numpy as np
import evaluate
import requests # Added for manual download
import os # Added for manual download
from datasets import Dataset, DatasetDict, ClassLabel, Features, Sequence, Value # Added Dataset, DatasetDict, etc.
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

# --- TASK 1: DATASET SELECTION (Manually downloading and parsing CoNLL-2003) ---

def download_file(url, filename):
    response = requests.get(url)
    response.raise_for_status() # Raise an exception for bad status codes
    with open(filename, 'wb') as f:
        f.write(response.content)
    print(f"Downloaded {filename}")

# URLs for CoNLL-2003 raw data files
urls = {
    'eng.train': 'https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.train',
    'eng.testa': 'https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testa',
    'eng.testb': 'https://raw.githubusercontent.com/synalp/NER/master/corpus/CoNLL-2003/eng.testb'
}

# Download the files
for filename, url in urls.items():
    download_file(url, filename)

def parse_conll_file(filepath):
    tokens = []
    ner_tags = []
    current_tokens = []
    current_ner_tags = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                # CoNLL-2003 format: Word   POS   ChunkTag   NERTag
                if len(parts) >= 4:
                    current_tokens.append(parts[0])
                    current_ner_tags.append(parts[3]) # Using NERTag
            else:
                # Empty line signifies end of a sentence
                if current_tokens:
                    tokens.append(current_tokens)
                    ner_tags.append(current_ner_tags)
                current_tokens = []
                current_ner_tags = []
    # Add the last sentence if the file doesn't end with an empty line
    if current_tokens:
        tokens.append(current_tokens)
        ner_tags.append(current_ner_tags)

    return {'tokens': tokens, 'ner_tags': ner_tags}

# Parse the downloaded files
train_data = parse_conll_file('eng.train')
validation_data = parse_conll_file('eng.testa')
test_data = parse_conll_file('eng.testb')

# Get all unique NER tags for mapping
all_ner_tags = sorted(list(set(tag for sublist in train_data['ner_tags'] for tag in sublist) |
                            set(tag for sublist in validation_data['ner_tags'] for tag in sublist) |
                            set(tag for sublist in test_data['ner_tags'] for tag in sublist)))

tag_to_id = {tag: i for i, tag in enumerate(all_ner_tags)}
id_to_tag = {i: tag for i, tag in enumerate(all_ner_tags)}

# Convert string tags to integer IDs
def encode_tags(tags):
    return [tag_to_id[tag] for tag in tags]

train_data['ner_tags_ids'] = [encode_tags(tags) for tags in train_data['ner_tags']]
validation_data['ner_tags_ids'] = [encode_tags(tags) for tags in validation_data['ner_tags']]
test_data['ner_tags_ids'] = [encode_tags(tags) for tags in test_data['ner_tags']]

# Create DatasetDict and assign to 'dataset' variable to match original code's variable name
dataset = DatasetDict({
    'train': Dataset.from_dict({
        'tokens': train_data['tokens'],
        'ner_tags': train_data['ner_tags_ids'] # Use ner_tags_ids
    }),
    'validation': Dataset.from_dict({
        'tokens': validation_data['tokens'],
        'ner_tags': validation_data['ner_tags_ids'] # Use ner_tags_ids
    }),
    'test': Dataset.from_dict({
        'tokens': test_data['tokens'],
        'ner_tags': test_data['ner_tags_ids'] # Use ner_tags_ids
    })
})

# Add feature information
ner_feature = ClassLabel(names=all_ner_tags)
features = Features({
    'tokens': Sequence(Value('string')),
    'ner_tags': Sequence(ner_feature)
})
dataset = dataset.map(lambda x: x, features=features) # Apply features to the new 'dataset'

# Identify label types (using the manually extracted 'all_ner_tags')
label_list = all_ner_tags # Now this is directly from manual parsing
id2label = id_to_tag
label2id = tag_to_id

print(f"Dataset: CoNLL-2003 (Manual Load) | Labels: {label_list}")

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# --- TASK 2: DATA PREPROCESSING & LABEL ALIGNMENT ---
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    # IMPORTANT: Change examples["chunk_tags"] to examples["ner_tags"]
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # Set labels for special tokens to -100 so they are ignored by the loss function
            if word_idx is None:
                label_ids.append(-100)
            # Only label the first token of a given word
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For subwords, we set the label to -100
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply preprocessing
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# --- TASK 3 & 4: MODEL SETUP & TRAINING ---
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Evaluation Metric (seqeval)
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Training Arguments
training_args = TrainingArguments(
    output_dir="./bert-chunking-results",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs', # Changed from TENSORBOARD_LOG_DIR back to logging_dir
)

# Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer)

# Trainer
# Updated Trainer without the 'tokenizer' keyword argument
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,  # The tokenizer is already inside this!
    compute_metrics=compute_metrics
)

# Start Training
trainer.train()

# --- TASK 5: EVALUATION ---
eval_results = trainer.evaluate()
print(f"Final Evaluation Results: {eval_results}")

# --- TASK 6: INFERENCE ---
# Creating a pipeline for easy prediction
token_classifier = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

custom_sentence = "John works at Google in California"
predictions = token_classifier(custom_sentence)

print("\n--- Inference Result ---")
for pred in predictions:
    print(f"Entity: {pred['word']} | Tag: {pred['entity_group']} | Score: {pred['score']:.4f}")

Downloaded eng.train
Downloaded eng.testa
Downloaded eng.testb


Map:   0%|          | 0/14987 [00:00<?, ? examples/s]

Map:   0%|          | 0/3466 [00:00<?, ? examples/s]

Map:   0%|          | 0/3684 [00:00<?, ? examples/s]

Dataset: CoNLL-2003 (Manual Load) | Labels: ['B-LOC', 'B-MISC', 'B-ORG', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final Evaluation Results: {'eval_loss': 0.04223483055830002, 'eval_precision': 0.9399328859060403, 'eval_recall': 0.9427802086839449, 'eval_f1': 0.941354394219459, 'eval_accuracy': 0.9887742836092908, 'eval_runtime': 6.2163, 'eval_samples_per_second': 557.57, 'eval_steps_per_second': 34.908, 'epoch': 3.0}

--- Inference Result ---
Entity: john | Tag: PER | Score: 0.9969
Entity: google | Tag: ORG | Score: 0.9906
Entity: california | Tag: LOC | Score: 0.9978
